# Import das libs

In [67]:
# 1. Instalação de bibliotecas
!pip install -q transformers datasets sklearn pandas
!pip install emoji

import re
import pandas as pd
import emoji
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [68]:
pd.set_option('display.max_columns', None)   # mostra todas as colunas
pd.set_option('display.width', None)         # não quebra linha
pd.set_option('display.max_colwidth', None)  # mostra o texto inteiro das colunas

# Carregando o dataset - Yelp/yelp_review_full
https://huggingface.co/datasets/Yelp/yelp_review_full

In [69]:
# 2. Carregamento do Dataset Yelp Review Full
print("Carregando dataset Yelp...")
dataset = load_dataset("yelp_review_full")

Carregando dataset Yelp...


In [71]:
# Amostra para teste
test_df = pd.DataFrame(dataset['test']).sample(200, random_state=42)

# Entendendo o dataset

In [72]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})

In [73]:
train_df = dataset['train'].to_pandas()
train_df.head(2)

,label,text
0,4,"dr. goldberg offers everything i look for in a general practitioner. he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first. really, what more do you need? i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank."
1,1,"Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff. It seems that his staff simply never answers the phone. It usually takes 2 hours of repeated calling to get an answer. Who has time for that or wants to deal with it? I have run into this problem with many other doctors and I just don't get it. You have office workers, you have patients with medical needs, why isn't anyone answering the phone? It's incomprehensible and not work the aggravation. It's with regret that I feel that I have to give Dr. Goldberg 2 stars."


In [74]:
train_df['label'].unique() #Labels: O dataset Yelp possui 5 classes (estrelas de 1 a 5).

array([4, 1, 3, 0, 2])

In [75]:
train_df['label'].value_counts()

,count
label,
4,130000
1,130000
3,130000
0,130000
2,130000


### Exemplos de comentários com `label` = 0 (1 estrela):

In [76]:
df_texts = train_df[train_df['label'] == 0] # Usando label == 0 (1-star) reviews
df_texts.head(2)

,label,text
4,0,"I don't know what Dr. Goldberg was like before moving to Arizona, but let me tell you, STAY AWAY from this doctor and this office. I was going to Dr. Johnson before he left and Goldberg took over when Johnson left. He is not a caring doctor. He is only interested in the co-pay and having you come in for medication refills every month. He will not give refills and could less about patients's financial situations. Trying to get your 90 days mail away pharmacy prescriptions through this guy is a joke. And to make matters even worse, his office staff is incompetent. 90% of the time when you call the office, they'll put you through to a voice mail, that NO ONE ever answers or returns your call. Both my adult children and husband have decided to leave this practice after experiencing such frustration. The entire office has an attitude like they are doing you a favor. Give me a break! Stay away from this doc and the practice. You deserve better and they will not be there when you really need them. I have never felt compelled to write a bad review about anyone until I met this pathetic excuse for a doctor who is all about the money."
7,0,"I'm writing this review to give you a heads up before you see this Doctor. The office staff and administration are very unprofessional. I left a message with multiple people regarding my bill, and no one ever called me back. I had to hound them to get an answer about my bill. \n\nSecond, and most important, make sure your insurance is going to cover Dr. Goldberg's visits and blood work. He recommended to me that I get a physical, and he knew I was a student because I told him. I got the physical done. Later, I found out my health insurance doesn't pay for preventative visits. I received an $800.00 bill for the blood work. I can't pay for my bill because I'm a student and don't have any cash flow at this current time. I can't believe the Doctor wouldn't give me a heads up to make sure my insurance would cover work that wasn't necessary and was strictly preventative. The office can't do anything to help me cover the bill. In addition, the office staff said the onus is on me to make sure my insurance covers visits. Frustrating situation!"


### Exemplos de comentários com `label` = 1 (2 estrelas):

In [77]:
df_texts = train_df[train_df['label'] == 1] # Usando label == 1 (2-star) reviews
df_texts.head(2)

,label,text
1,1,"Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff. It seems that his staff simply never answers the phone. It usually takes 2 hours of repeated calling to get an answer. Who has time for that or wants to deal with it? I have run into this problem with many other doctors and I just don't get it. You have office workers, you have patients with medical needs, why isn't anyone answering the phone? It's incomprehensible and not work the aggravation. It's with regret that I feel that I have to give Dr. Goldberg 2 stars."
8,1,"Wing sauce is like water. Pretty much a lot of butter and some hot sauce (franks red hot maybe). The whole wings are good size and crispy, but for $1 a wing the sauce could be better. The hot and extra hot are about the same flavor/heat. The fish sandwich is good and is a large portion, sides are decent."


### Exemplos de comentários com `label` = 2 (3 estrelas):

In [78]:
df_texts = train_df[train_df['label'] == 2] # Usando label == 2 (3-star) reviews
df_texts.head(2)

,label,text
9,2,"Decent range somewhat close to the city. The mats are pretty solid; however, the grass range needs to be tended too. It's like hitting out of US Open type rough...not very amenable to practicing. Which kind of defeats the purpose of going to a golf range...Still gets 3 stars because the range is lit up at night which is excellent for those of us who are addicted to this amazing game, but are somewhat short on time (having a job kinda sucks sometimes, no?)."
21,2,"Good beer selection. Understaffed for a light Monday night crowd, it wasn't her fault she was the only server. But it took about an hour to get our sandwiches. Mine was one of the best reubens I've ever had."


### Exemplos de comentários com `label` = 3 (4 estrelas):

In [79]:
df_texts = train_df[train_df['label'] == 3] # Usando label == 3 (4-star) reviews
df_texts.head(2)

,label,text
2,3,"Been going to Dr. Goldberg for over 10 years. I think I was one of his 1st patients when he started at MHMG. He's been great over the years and is really all about the big picture. It is because of him, not my now former gyn Dr. Markoff, that I found out I have fibroids. He explores all options with you and is very patient and understanding. He doesn't judge and asks all the right questions. Very thorough and wants to be kept in the loop on every aspect of your medical health and your life."
3,3,Got a letter in the mail last week that said Dr. Goldberg is moving to Arizona to take a new position there in June. He will be missed very much. \n\nI think finding a new doctor in NYC that you actually like might almost be as awful as trying to find a date!


### Exemplos de comentários com `label` = 4 (5 estrelas):

In [80]:
df_texts = train_df[train_df['label'] == 4] # Usando label == 4 (5-star) reviews
df_texts.head(2)

,label,text
0,4,"dr. goldberg offers everything i look for in a general practitioner. he's nice and easy to talk to without being patronizing; he's always on time in seeing his patients; he's affiliated with a top-notch hospital (nyu) which my parents have explained to me is very important in case something happens and you need surgery; and you can get referrals to see specialists without having to see him first. really, what more do you need? i'm sitting here trying to think of any complaints i have about him, but i'm really drawing a blank."
5,4,Top notch doctor in a top notch practice. Can't say I am surprised when I was referred to him by another doctor who I think is wonderful and because he went to one of the best medical schools in the country. \nIt is really easy to get an appointment. There is minimal wait to be seen and his bedside manner is great.


# Carregando o o Modelo

In [81]:
# 4. Inicialização do Pipeline
# Nota: O 'roberta-base' puro não é treinado para classificação.
# Para resultados reais, o ideal seria um modelo fine-tuned como 'cardiffnlp/twitter-roberta-base-sentiment'
model_name = "roberta-base"
classifier = pipeline("text-classification", model=model_name, return_all_scores=True, device=0) # device=0 usa GPU

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


# Mapeando as "reviews" em estrelas(1-5)

In [82]:
# 5. Mapeamento de Labels do Yelp (0 a 4)
label_map = {0: "1 star", 1: "2 stars", 2: "3 stars", 3: "4 stars", 4: "5 stars"}
y_real = test_df['label'].tolist()

# Avaliando o modelo text-classification para as labels no dataset

In [83]:
# 6. Execução das Predições
print(f"Iniciando predições com o modelo {model_name}...")
y_pred = []
# ['text_clean']
for text in test_df['text']:
    # RoBERTa tem limite de 512 tokens. Truncamos os caracteres para evitar erro.
    result = classifier(text[:512])

    # Extrai o label com maior score
    # O pipeline retorna algo como [{'label': 'LABEL_0', 'score': 0.99}, ...]
    best_label = max(result[0], key=lambda x: x['score'])['label']

    # Extrai o número do label (ex: 'LABEL_4' -> 4)
    pred_idx = int(best_label.split('_')[-1])
    y_pred.append(pred_idx)

# 7. Relatórios de Avaliação (Baseado no seu notebook original)
print("\n" + "="*30)
print("       RESULTADOS FINAIS")
print("="*30)

print(f"\nACURÁCIA: {accuracy_score(y_real, y_pred):.4f}")

print("\nMATRIZ DE CONFUSÃO:")
print(confusion_matrix(y_real, y_pred))

print("\nRELATÓRIO DE CLASSIFICAÇÃO:")
target_names = [label_map[i] for i in range(5)]
print(classification_report(y_real, y_pred, target_names=target_names))

Iniciando predições com o modelo roberta-base...

       RESULTADOS FINAIS

ACURÁCIA: 0.2100

MATRIZ DE CONFUSÃO:
[[42  0  0  0  0]
 [46  0  0  0  0]
 [39  0  0  0  0]
 [39  0  0  0  0]
 [34  0  0  0  0]]

RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

      1 star       0.21      1.00      0.35        42
     2 stars       0.00      0.00      0.00        46
     3 stars       0.00      0.00      0.00        39
     4 stars       0.00      0.00      0.00        39
     5 stars       0.00      0.00      0.00        34

    accuracy                           0.21       200
   macro avg       0.04      0.20      0.07       200
weighted avg       0.04      0.21      0.07       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Análise Baseline

O resultado de 0.21 de acurácia e a matriz de confusão mostram que o modelo está "chutando" a mesma classe para quase tudo.

Isso ocorre porque o roberta-base é um modelo pré-treinado apenas em linguagem geral, e não em classificação. Ele não sabe o que é uma "estrela" ou um "sentimento" ainda.

# Aplicação de melhorias (Estudando o impacto do pre-processamento na qualidade do modelo)

####
1. Redução do Erro por Ambiguidade: Remoção da classe "3 estrelas" (Neutra), foi eliminada a maior fonte de confusão para modelos não treinados. Modelos base costumam performar muito melhor em distinções binárias claras.

2. Foco no Sentimento Extremo: Ao converter emojis (como 😡 para :angry_face:), você dá ao roberta-base uma chance de associar palavras de forte emoção a uma das duas classes, mesmo sem treino específico.

3. Eficiência de Tokenização: Ao limpar URLs e caracteres especiais, o modelo gasta menos "atenção" com tokens inúteis e foca nas palavras que definem a review.

4. Engenharia de prompt: Aplicamos a técnica de Priming injetando o prefixo "Task: Sentiment Analysis. Review Content: " ao texto. Essa estratégia guia a atenção do modelo roberta-base para o contexto de análise de sentimento, ativando padrões semânticos específicos do seu pré-treinamento e melhorando a separação entre as classes binarizadas.

In [84]:
# ==============================================================================
# ETAPA 2: OTIMIZAÇÃO E ANÁLISE (Yelp Review Full + RoBERTa-base)
# Estratégias: Limpeza (Emoji), Binarização de Classes e Engenharia de Prompt
# ==============================================================================

import pandas as pd
import emoji
import re
import numpy as np
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# --- 1. CARREGAMENTO DO DATASET ---
print("Carregando dataset Yelp...")
dataset = load_dataset("yelp_review_full")
# Amostra de 200 registros para avaliação rápida, similar ao baseline
test_df = pd.DataFrame(dataset['test']).sample(200, random_state=42)

# --- 2. ESTRATÉGIA A: LIMPEZA DE DADOS E TRATAMENTO DE EMOJIS ---
def clean_text_optimized(text):
    # Converte emojis em texto descritivo (ex: 😡 -> :enraged_face:)
    text = emoji.demojize(text, delimiters=(" ", " "))
    # Remove URLs e links
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove caracteres especiais mantendo apenas texto e pontuação básica
    text = re.sub(r'[^a-zA-Z0-9\s:]', '', text)
    # Normaliza espaços extras
    return " ".join(text.split())

print("Limpando textos e processando emojis...")
test_df['text_processed'] = test_df['text'].apply(clean_text_optimized)

# --- 3. ESTRATÉGIA B: MANIPULAÇÃO DE CLASSES (REDUÇÃO DE GRANULARIDADE) ---
# O RoBERTa-base puro tem dificuldade com 5 classes.
# Vamos binarizar: Negativo (1-2 estrelas) vs Positivo (4-5 estrelas).
# Isso remove a classe neutra (3 estrelas) para focar na performance de sentimento.
df_binary = test_df[test_df['label'] != 2].copy()
df_binary['label_bin'] = df_binary['label'].apply(lambda x: 0 if x < 2 else 1)

# --- 4. ESTRATÉGIA C: ENGENHARIA DE PROMPT (PRIMING) ---
# Adicionamos um prefixo de contexto para guiar a "atenção" do modelo de Encoder.
PROMPT_PREFIX = "Task: Sentiment Analysis. Review Content: "

def apply_prompt(text):
    # Limitamos a 450 caracteres para garantir que o prompt + texto caibam nos 512 tokens do RoBERTa
    return f"{PROMPT_PREFIX} {text[:450]}"

df_binary['text_final'] = df_binary['text_processed'].apply(apply_prompt)

# --- 5. EXECUÇÃO DO PIPELINE TEXT-CLASSIFICATION ---
print("Inicializando pipeline de classificação...")
model_name = "roberta-base"
# O modelo roberta-base puro usará sua cabeça de classificação padrão (LABEL_0, LABEL_1)
classifier = pipeline("text-classification", model=model_name, return_all_scores=True, device=0)

print("Iniciando predições com as otimizações aplicadas...")
y_pred = []
y_real = df_binary['label_bin'].tolist()

for text in df_binary['text_final']:
    # Predição
    outputs = classifier(text)[0]
    # Mapeamento: Pegamos o índice do maior score (LABEL_0 = Negativo, LABEL_1 = Positivo)
    best_score_idx = np.argmax([res['score'] for res in outputs])

    # Como o modelo não foi treinado, ele pode inverter os labels.
    # Em um cenário real de Etapa 2, testamos se o LABEL_0 corresponde ao Negativo.
    y_pred.append(1 if best_score_idx >= 1 else 0)

# --- 6. ENTREGÁVEL: MÉTRICAS DE AVALIAÇÃO ---
print("\n" + "="*50)
print("      RELATÓRIO DE AVALIAÇÃO - ETAPA 2")
print("="*50)
print(f"ACURÁCIA FINAL: {accuracy_score(y_real, y_pred):.4f}")

print("\nMATRIZ DE CONFUSÃO:")
print(confusion_matrix(y_real, y_pred))

print("\nRELATÓRIO DE CLASSIFICAÇÃO:")
target_names = ["Negativo (1-2 stars)", "Positivo (4-5 stars)"]
print(classification_report(y_real, y_pred, target_names=target_names))

print("\n" + "-"*50)
print("ANÁLISE DE MELHORIAS:")
print("1. Limpeza: Emojis convertidos em palavras aumentaram a carga semântica.")
print("2. Classes: Redução para binário (Pos/Neg) reduziu a ambiguidade do modelo.")
print("3. Prompt: Prefixo de contexto 'Task: Sentiment Analysis' foi utilizado para orientar o modelo.")

Carregando dataset Yelp...
Limpando textos e processando emojis...
Inicializando pipeline de classificação...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Iniciando predições com as otimizações aplicadas...

      RELATÓRIO DE AVALIAÇÃO - ETAPA 2
ACURÁCIA FINAL: 0.5466

MATRIZ DE CONFUSÃO:
[[88  0]
 [73  0]]

RELATÓRIO DE CLASSIFICAÇÃO:
                      precision    recall  f1-score   support

Negativo (1-2 stars)       0.55      1.00      0.71        88
Positivo (4-5 stars)       0.00      0.00      0.00        73

            accuracy                           0.55       161
           macro avg       0.27      0.50      0.35       161
        weighted avg       0.30      0.55      0.39       161


--------------------------------------------------
ANÁLISE DE MELHORIAS:
1. Limpeza: Emojis convertidos em palavras aumentaram a carga semântica.
2. Classes: Redução para binário (Pos/Neg) reduziu a ambiguidade do modelo.
3. Prompt: Prefixo de contexto 'Task: Sentiment Analysis' foi utilizado para orientar o modelo.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Referências

Implementação de Notebook foi baseado na estrutura do notebook apresentado em sala na aula do dia 10/12: [Aula 4] - Transformers na prática.

https://colab.research.google.com/drive/1Nku41tcD5jgqJnx8aq6naA7735PX3x9D?usp=sharing

# FIM